In [1]:
from transformers import MarianMTModel, MarianTokenizer
import torch

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load translation model and tokenizer
def load_translation_model(model_name):
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    return tokenizer, model

In [6]:
# Translate a sentence
def translate(texts, tokenizer, model):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

In [7]:
# Models
cy2en_tokenizer, cy2en_model = load_translation_model("Helsinki-NLP/opus-mt-cy-en")
en2cy_tokenizer, en2cy_model = load_translation_model("Helsinki-NLP/opus-mt-en-cy")

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\transformers\models\marian\tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\c24082331\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-cy. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either 

In [8]:
# Back translation function
def back_translate_welsh(text):
    # Welsh to English
    english = translate([text], cy2en_tokenizer, cy2en_model)[0]
    # English to Welsh
    back_translated = translate([english], en2cy_tokenizer, en2cy_model)[0]
    return back_translated, english

In [11]:
welsh_texts = ["Dw i'n hoffi coffi.", "Mae hi'n braf heddiw."]

for text in welsh_texts:
    bt_welsh, mid_english = back_translate_welsh(text)
    print(f"Original: {text}")
    print(f"English: {mid_english}")
    print(f"Back-Translated: {bt_welsh}")
    print("-" * 40)

Original: Dw i'n hoffi coffi.
English: I like coffee.
Back-Translated: Rwy'n hoff o goffi.
----------------------------------------
Original: Mae hi'n braf heddiw.
English: She's nice today.
Back-Translated: Mae hi'n braf heddiw.
----------------------------------------


In [12]:
import sacrebleu

original = ["Dw i'n hoffi coffi."]
back_translated = ["Rwy'n hoffi coffi."]

bleu = sacrebleu.corpus_bleu(back_translated, [original])
chrf = sacrebleu.corpus_chrf(back_translated, [original])

print(f"BLEU: {bleu.score:.2f}")
print(f"chrF: {chrf.score:.2f}")


BLEU: 46.31
chrF: 78.45


In [17]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('distiluse-base-multilingual-cased')

def semantic_similarity(s1, s2):
    emb1 = model.encode(s1, convert_to_tensor=True)
    emb2 = model.encode(s2, convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()

sim = semantic_similarity("Dw i'n hoffi coffi.", "Rwy'n hoffi coffi.")
print(f"Cosine similarity: {sim:.4f}")

Cosine similarity: 0.7593
